# Exercises: Swapping embeddings

Machine Learning for Digital Scholarly Editions. Build your BERTopic pipeline.

In the main notebook we swapped in a bigger Sentence Transformers model on the DH abstracts. Here we try a language swap instead, on a different corpus: the **Hugo Schuchardt Archive letters**, a collection of historical correspondence that mixes several languages in the same file. Work through the exercises in order, checking the main notebook or the [BERTopic documentation](https://maartengr.github.io/BERTopic/) when you get stuck.

Every model in this notebook uses `verbose=True` and a multilingual `vectorizer_model` (built in Exercise 1) as a standing default, the same way you'd always want stopwords handled in a real project, not just for the one exercise that happens to be about them.

All solutions are in the separate `02_swapping-embeddings_solutions.ipynb` notebook. Try each exercise yourself first.

## Setup

If you're on **Google Colab**, run the cell below to mount your Drive and move into the materials folder.

If you're running **locally**, skip/comment out the cell below and just make sure `letters_full.csv` is in your working directory.

In [ ]:
# Colab only
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/dse-ml-2026/materials/TODO-path/

### Exercise 1. Build a genuinely multilingual sample, and a multilingual vectorizer

**The sample.** Just picking 2000 rows completely at random risks diluting whatever cross-lingual thematic overlap this corpus actually has. Most letters are in German, so a blind random sample mostly tests "does the model handle German", not "does it find themes shared across languages". The `keywords` column gives us a way to check for that overlap directly: it holds archivist-assigned subject tags, semicolon-separated, and the same subject often gets tagged on letters written in different languages.

1. Load `letters_full.csv` into `df`.
2. Find which individual `keywords` values appear in **3 or more different languages** elsewhere in the corpus. (Split `keywords` on `;`; for each resulting keyword, count how many distinct `language` values it's associated with across the whole dataset.)
3. Filter `df` down to rows whose `keywords` field contains at least one of those cross-lingual keywords.
4. From that filtered set, take an **equal-sized sample from each of the six languages with enough data to support it**: `de`, `fr`, `it`, `pt`, `es`, `en` (about 333 rows each, `random_state=42`). Combine them and store the texts in `documents`.
5. Print the language breakdown of your final sample with `.value_counts()` to confirm it's actually balanced.

**The vectorizer.** Every topic word list you'll look at in this notebook would otherwise be dominated by function words (`de`, `ich`, `que`, `la`, ...), which makes it impossible to judge whether a topic is genuinely coherent, so build this once now and reuse it in every model you fit below:

6. Build a combined stopword list covering all six languages in the sample, using NLTK's `stopwords` corpus (`german`, `french`, `italian`, `portuguese`, `spanish`, `english`).
7. Create `vectorizer_model = CountVectorizer(stop_words=...)` with that combined list, and keep it around, you'll pass it to every `BERTopic(...)` you create from here on.

In [ ]:
# 1. load letters_full.csv into df


# 2. find keywords that appear in 3+ languages


# 3. filter df to rows tagged with at least one cross-lingual keyword


# 4. sample ~333 rows from each of the 6 languages with enough data -> documents


# 5. print the language breakdown of the final sample


# 6. build a combined multilingual stopword list with nltk


# 7. create vectorizer_model with that stopword list


### Exercise 2. Run the default pipeline as a baseline

Create a `BERTopic` model exactly like you did for the DH abstracts (`language="english"`), but this time also pass your `vectorizer_model` from Exercise 1. Fit it on `documents`, then look at `get_topic_info()`.

This is your baseline: the English-tuned pipeline applied to a corpus that is only partly English.

In [ ]:
# create and fit a baseline topic_model_baseline, with vectorizer_model

# inspect get_topic_info()


### Exercise 3. Quantify how many documents ended up as outliers

Topic `-1` is the outlier bucket (see notebook 1, Exercise 3). Using `get_topic_info()`, compute **what percentage of all documents** landed in topic `-1` for the baseline model. Keep the number, you'll compare it to the multilingual runs in Exercises 5 and 7.

(Hint: the `Count` column of `get_topic_info()` gives you the size of each topic, and it sums to the total number of documents.)

In [ ]:
# compute the percentage of documents in topic -1 for the baseline model


### Exercise 4. Try the multilingual shortcut

`BERTopic` accepts a `language` argument other than `"english"` for exactly this situation, mixed-language text. Find the right value in the [BERTopic documentation](https://maartengr.github.io/BERTopic/), create a `topic_model_multilingual` with it (plus `vectorizer_model`, as always from here on), and fit it on `documents`.

In [ ]:
# create and fit topic_model_multilingual using the language argument and vectorizer_model


### Exercise 5. Compare

Repeat what you did in Exercise 3, this time for `topic_model_multilingual`. Did the percentage of outliers go up or down compared to the baseline? Also compare a few topics' top words between the two models using `get_topic()`, are they more coherent (words that clearly belong together) in one of the two? Don't assume the multilingual model automatically wins, look at your actual numbers.

In [ ]:
# compute the percentage of documents in topic -1 for the multilingual model


# compare a few topics between the baseline and multilingual models


### Exercise 6. Reproduce it by hand

`language="multilingual"` is itself just a shortcut: under the hood, BERTopic uses `paraphrase-multilingual-MiniLM-L12-v2` as the embedding model. Confirm that yourself:

1. Load that same model with `SentenceTransformer`.
2. Pass it as `embedding_model` to a new `BERTopic` model, along with `vectorizer_model`.
3. Fit `topic_model_manual` on `documents`, and compare its `get_topic_info()` to `topic_model_multilingual` from Exercise 4.

In [ ]:
# 1 & 2. load the SentenceTransformer, create topic_model_manual with it and vectorizer_model


# 3. fit it on documents, compare to topic_model_multilingual


### Exercise 7. Use a model the shortcut can't give you

`language="multilingual"` only ever gets you that one specific model. `embedding_model=` is what actually lets you use a *different* one. The main notebook's own embeddings swap on the DH abstracts went from the default English model (`all-MiniLM-L6-v2`) up to a larger one (`all-mpnet-base-v2`), there's a multilingual counterpart to that same upgrade: `paraphrase-multilingual-mpnet-base-v2`.

Load it, fit a `topic_model_larger` on `documents` (with `vectorizer_model`, as always), and compare it against both `topic_model_baseline` and `topic_model_multilingual`. Compare **both** the outlier % and the number of non-outlier topics found (`get_topic_info()` has a row per topic, so counting rows where `Topic != -1` gives you this), not just one or the other: a model can post a great outlier percentage by collapsing everything into one or two giant undifferentiated topics rather than by genuinely finding structure. Does the larger model actually do better here, on both measures, and do its topics actually read as coherent now that you can see the real words?

In [ ]:
# load paraphrase-multilingual-mpnet-base-v2, fit topic_model_larger with vectorizer_model


# compare outlier % AND number of topics found, against topic_model_baseline and topic_model_multilingual
